In [26]:
import sys
import os

# FORCE the correct project root
sys.path.insert(0, r"C:\Users\Pablo Miller\proyectos\pabs-nba-analytics-dashboard")

print(sys.path[0])


# Detect the real project root (two levels up if needed)
current = os.getcwd()
print("Notebook working directory:", current)

# Try parent folder
parent = os.path.abspath(os.path.join(current, ".."))
print("Parent folder:", parent)

# Try grandparent folder
grandparent = os.path.abspath(os.path.join(current, "..", ".."))
print("Grandparent folder:", grandparent)

# Add both to sys.path
sys.path.append(parent)
sys.path.append(grandparent)

print("Final sys.path:")
for p in sys.path:
    print(" -", p)


C:\Users\Pablo Miller\proyectos\pabs-nba-analytics-dashboard
Notebook working directory: c:\Users\Pablo Miller\proyectos\pabs-nba-analytics-dashboard\notebooks\reference
Parent folder: c:\Users\Pablo Miller\proyectos\pabs-nba-analytics-dashboard\notebooks
Grandparent folder: c:\Users\Pablo Miller\proyectos\pabs-nba-analytics-dashboard
Final sys.path:
 - C:\Users\Pablo Miller\proyectos\pabs-nba-analytics-dashboard
 - C:\Users\Pablo Miller\AppData\Local\Python\pythoncore-3.12-64\python312.zip
 - C:\Users\Pablo Miller\AppData\Local\Python\pythoncore-3.12-64\DLLs
 - C:\Users\Pablo Miller\AppData\Local\Python\pythoncore-3.12-64\Lib
 - C:\Users\Pablo Miller\AppData\Local\Python\pythoncore-3.12-64
 - c:\Users\Pablo Miller\proyectos\pabs-nba-analytics-dashboard\.venv
 - 
 - c:\Users\Pablo Miller\proyectos\pabs-nba-analytics-dashboard\.venv\Lib\site-packages
 - c:\Users\Pablo Miller\proyectos\pabs-nba-analytics-dashboard\notebooks
 - c:\Users\Pablo Miller\proyectos\pabs-nba-analytics-dashboard\

In [27]:
import pandas as pd
import matplotlib.pyplot as plt

from src.db import run_query
from src.colors import get_primary, get_secondary
from src.metrics import add_advanced_metrics

plt.style.use("ggplot")

In [28]:
seasons = run_query("SELECT DISTINCT SEASON FROM season_stats ORDER BY SEASON")
seasons

DatabaseError: Execution failed on sql 'SELECT DISTINCT SEASON FROM season_stats ORDER BY SEASON': no such table: season_stats

In [ ]:
# Which teams attempted the most threes?

query = """
SELECT TEAM_ABBREVIATION, SUM(FG3A * GP) AS TOTAL_THREES_ATTEMPTED
FROM season_stats
GROUP BY TEAM_ABBREVIATION
ORDER BY TOTAL_THREES_ATTEMPTED DESC
LIMIT 5;
"""

df = pd.read_sql_query(query, conn)
df

,TEAM_ABBREVIATION,TOTAL_THREES_ATTEMPTED
0,POR,3674.0
1,CHA,3591.3
2,CLE,3566.5
3,ATL,3548.8
4,GSW,3451.0


In [ ]:
query = """
SELECT PLAYER_NAME, PTS, GP, TEAM_ABBREVIATION
FROM season_stats
WHERE SEASON = ?
ORDER BY PTS DESC
LIMIT 10
"""

df_top_scorers = run_query(query, (selected_season,))
df_top_scorers

,PLAYER_NAME,AST
0,Nikola Jokić,10.7
1,Cade Cunningham,9.9
2,Josh Giddey,9.1
3,Luka Dončić,8.3
4,Ja Morant,8.1
5,James Harden,8.0
6,Trae Young,8.0
7,Jalen Johnson,7.9
8,Andrew Nembhard,7.7
9,Kevin Porter Jr.,7.4


In [ ]:
colors = [get_primary(t) for t in df_top_scorers["TEAM_ABBREVIATION"]]
edges = [get_secondary(t) for t in df_top_scorers["TEAM_ABBREVIATION"]]

plt.figure(figsize=(10,6))
plt.bar(df_top_scorers["PLAYER_NAME"], df_top_scorers["PTS"],
        color=colors, edgecolor=edges, linewidth=3)
plt.xticks(rotation=45)
plt.title(f"Top Scorers — {selected_season}")
plt.show()

,PLAYER_NAME,PPG,POINTS_TOTAL
0,Cooper Flagg,21.0,1470.0
1,Kon Knueppel,18.5,1498.5
2,VJ Edgecombe,16.0,1200.0
3,Jeremiah Fears,14.3,1172.6
4,Ace Bailey,13.8,993.6
5,Cedric Coward,13.6,843.2
6,Maxime Raynaud,12.5,925.0
7,Tre Johnson,12.2,732.0
8,Dylan Harper,11.8,814.2
9,Derik Queen,11.7,947.7


In [ ]:
query = """
SELECT TEAM_ABBREVIATION, SUM(FG3A * GP) AS TOTAL_3PA
FROM season_stats
WHERE SEASON = ?
GROUP BY TEAM_ABBREVIATION
ORDER BY TOTAL_3PA DESC
"""

df_3pa = run_query(query, (selected_season,))
df_3pa

,PLAYER_NAME,FG_PCT,MIN
0,Jericho Sims,0.784,19.7
1,Jaxson Hayes,0.756,18.3
2,Ryan Kalkbrenner,0.753,21.4
3,Mitchell Robinson,0.723,19.6
4,Robert Williams III,0.708,17.1
5,Walker Kessler,0.703,30.8
6,Jakob Poeltl,0.700,25.0
7,Rudy Gobert,0.682,31.3
8,Goga Bitadze,0.676,15.2
9,Deandre Ayton,0.671,27.2


In [ ]:
colors = [get_primary(t) for t in df_3pa["TEAM_ABBREVIATION"]]
edges = [get_secondary(t) for t in df_3pa["TEAM_ABBREVIATION"]]

plt.figure(figsize=(12,6))
plt.bar(df_3pa["TEAM_ABBREVIATION"], df_3pa["TOTAL_3PA"],
        color=colors, edgecolor=edges, linewidth=3)
plt.xticks(rotation=45)
plt.title(f"Team 3PA — {selected_season}")
plt.show()

In [ ]:
query = """
SELECT PLAYER_NAME, TEAM_ABBREVIATION,
       PTS, FGA, FTA, FGM, FG3M, AST, TOV
FROM season_stats
WHERE SEASON = ?
AND MIN >= 30
"""

df_adv = run_query(query, (selected_season,))
df_adv = add_advanced_metrics(df_adv)
df_adv.head()

In [ ]:
df_ts = df_adv.sort_values("TS_PCT", ascending=False).head(10)

colors = [get_primary(t) for t in df_ts["TEAM_ABBREVIATION"]]
edges = [get_secondary(t) for t in df_ts["TEAM_ABBREVIATION"]]

plt.figure(figsize=(10,6))
plt.bar(df_ts["PLAYER_NAME"], df_ts["TS_PCT"],
        color=colors, edgecolor=edges, linewidth=3)
plt.xticks(rotation=45)
plt.title(f"Top TS% — {selected_season}")
plt.show()